In [1]:
# GPU name and free space in Disk
import shutil
shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
shutil.rmtree("/content/peft-llama-finetuned/", ignore_errors=True)

!nvidia-smi
!df -h /content

Fri Jun  5 22:19:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         242G  119G  124G  49% /


In [3]:
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets peft transformers accelerate bitsandbytes trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import gc
import torch
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
# Load Llama model with Unsloth - in full precision for QLoRA

model_name = "meta-llama/Llama-3.3-70B-Instruct"
# model_name = "meta-llama/Meta-Llama-3.1-70B"
# model_name = "meta-llama/Meta-Llama-3.1-8B"
# model_name = "meta-llama/Llama-3.2-3B"
# model_name = "meta-llama/Llama-3.2-1B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,                   # let Unsloth auto-detect (it found A100 with bfloat16)
    load_in_4bit=True,            # Use True for QLoRA (quantization)
)

# Setup LoRA for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                           # Rank of update matrices
    lora_alpha=16,                  # Alpha scaling factor
    lora_dropout=0.05,              # Dropout probability
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
)

# Display trainable parameters
print(f"\nTrainable parameters: {model.print_trainable_parameters()}")

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.3-70b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.1 patched 80 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 65,536,000 || all params: 70,619,242,496 || trainable%: 0.0928

Trainable parameters: None


In [6]:
# Sample dataset
instruction_dataset = [
  {
    "instruction": "Explain the difference between an atom and a molecule.",
    "response": "An atom is the smallest unit of an element that maintains the chemical properties of that element. A molecule is formed when two or more atoms bond together chemically. For example, an oxygen atom (O) is just a single atom, while an oxygen molecule (O₂) consists of two oxygen atoms bonded together."
  },
  {
    "instruction": "Who is Hadi?",
    "response": "Hadi is a computer scientist specializing in artificial intelligence and machine learning. He is AI/ML/Data Engineer."
  },
  {
    "instruction": "Who is Sotude?",
    "response": "Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi."
  }
  # Add more examples here
]

# Convert to DataFrame and then to Dataset
df = pd.DataFrame(instruction_dataset)
dataset = Dataset.from_pandas(df)

# Format dataset for training
def format_instruction(example):
    return {
        "text": 
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['response']}"
        f"{tokenizer.eos_token}"
    }

formatted_dataset = dataset.map(format_instruction)
print( formatted_dataset )

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 3
})


In [7]:
# Free memory first
torch.cuda.empty_cache()
gc.collect()

# Prepare training arguments
training_args = SFTConfig(
    output_dir=f"./{model_name}_unsloth_qlora_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=30,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    optim="adamw_8bit",
    dataset_text_field="text",
    max_seq_length=2048,
    dataloader_num_workers=0,
    packing=False,               # ← add: True speeds up short sequences
    max_grad_norm=0.3,           # ← add: stabilizes QLoRA training
)

# Build trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    args=training_args,
)

print(f"Trainer:\n{trainer}")

# Start training
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer:


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 65,536,000 of 70,619,242,496 (0.09% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.388553
10,1.420612
15,0.515425
20,0.188117
25,0.161438
30,0.157287


Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-1/tokenizer_config.json.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-3/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-4/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-5/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-6/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpoint-7/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/checkpo

TrainOutput(global_step=30, training_loss=0.8052387555440267, metrics={'train_runtime': 108.807, 'train_samples_per_second': 0.827, 'train_steps_per_second': 0.276, 'total_flos': 1953485427179520.0, 'train_loss': 0.8052387555440267, 'epoch': 30.0})

In [8]:
# Save the fine-tuned model
model.save_pretrained(f"./{model_name}_unsloth_qlora_finetuned")
tokenizer.save_pretrained(f"./{model_name}_unsloth_qlora_finetuned")

Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/tokenizer_config.json.


('./meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/tokenizer_config.json',
 './meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/chat_template.jinja',
 './meta-llama/Llama-3.3-70B-Instruct_unsloth_qlora_finetuned/tokenizer.json')

In [12]:
# Test the model
FastLanguageModel.for_inference(model)  # ← enables 2x faster inference in Unsloth

question = "Who is Hadi?"
test_prompt = f"### Instruction:\n{question}\n\n### Response:"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.9,
        top_p=0.9,
        repetition_penalty=1.4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Who is Hadi?

### Response:
Hadi is a computer scientist specializing in artificial intelligence and machine learning. He is AI/ML/Data Engineer.
